# Post-Training Delivery Instructor Notebook

This notebook teaches you how to move from **trained model artifacts** to a **production deployment** using CI/CD, containers, and cloud delivery.

Learning goals:
- Understand post-training handoff contracts
- Build and test an inference API
- Containerize and run locally
- Implement CI and CD pipelines
- Prepare for secure cloud deployment and rollback

How to use with this repo:
- Read each theory section
- Complete starter files that contain:
  - `#start code here`
  - `#send code here`
- Run the commands and mini exercises after each section

## 1) Set Up the Learning Workspace and Toolchain

Post-training reproducibility means: if you train once and deploy many times, each runtime should behave identically.

Why this matters:
- Dependency drift changes predictions
- Different artifact files create silent regressions
- Local-vs-cloud runtime mismatches cause deployment failures

Reproducibility checklist:
- Pin Python packages
- Hash model artifacts
- Track model metadata and schema
- Use containers for runtime consistency

In [ ]:
import os, platform, subprocess

print('Python version:', platform.python_version())
print('OS:', platform.platform())
print('Workspace:', os.getcwd())

# Run manually in terminal too:
# git --version
# docker --version
# pip --version

## 2) Create Project Scaffold Files with Starter Markers

You will complete code only between marker lines in each file.

Target files:
- `app/main.py`
- `app/model_loader.py`
- `tests/test_api.py`
- `requirements.txt`
- `Dockerfile`
- `docker-compose.yml`
- `.github/workflows/ci.yml`
- `.github/workflows/cd.yml`
- `cloud/deploy/cloudrun-service.yaml`
- `cloud/config/dev.env`, `cloud/config/staging.env`, `cloud/config/prod.env`

Knowledge check:
1. Why are starter markers useful for guided learning?
2. Which files are runtime concerns vs pipeline concerns?

## 3) Package Post-Training Artifacts for Reproducible Inference

Post-training handoff contract:
- Training produces artifact files + metadata
- Serving consumes exact artifact versions
- CI validates schema and checksum before deploy

Manifest fields to track:
- model_version
- dataset_version
- metrics (accuracy/F1/etc.)
- input_schema_version
- artifact_checksum
- created_at

This contract prevents "works in notebook, fails in prod" incidents.

In [ ]:
import hashlib, json
from pathlib import Path

artifact_path = Path('artifacts/model.bin')
manifest_path = Path('artifacts/manifest.json')
artifact_path.parent.mkdir(parents=True, exist_ok=True)

if not artifact_path.exists():
    artifact_path.write_bytes(b'placeholder-model-bytes')

checksum = hashlib.sha256(artifact_path.read_bytes()).hexdigest()
manifest = {
    'model_version': 'v0.1.0',
    'dataset_version': 'demo-ds-001',
    'metrics': {'accuracy': 0.91},
    'input_schema_version': '1.0.0',
    'artifact_checksum': checksum,
}
manifest_path.write_text(json.dumps(manifest, indent=2))
print('wrote', manifest_path)
print('sha256:', checksum)

## 4) Build a Minimal Inference API Service

You will implement a small API endpoint and a model loading function.

Online inference latency decomposition:
$$
T_{total}=T_{deserialize}+T_{predict}+T_{serialize}
$$

Performance idea:
- Cache model at startup
- Validate input early
- Keep response schema minimal

Knowledge check:
1. Which latency term is usually largest for your model type?
2. Why does schema validation belong at the API boundary?

In [ ]:
# Tiny latency budget simulation
import random

t_deserialize = random.uniform(1, 3)
t_predict = random.uniform(4, 12)
t_serialize = random.uniform(1, 2)
t_total = t_deserialize + t_predict + t_serialize
print(f'T_deserialize={t_deserialize:.2f}ms, T_predict={t_predict:.2f}ms, T_serialize={t_serialize:.2f}ms')
print(f'T_total={t_total:.2f}ms')

## 5) Write Unit and Integration Tests for Model Serving

Test pyramid for serving systems:
- Many fast unit tests (model logic, schema checks)
- Fewer integration tests (API + app wiring)
- Minimal end-to-end tests

Why this matters post-training:
- Model behavior can regress silently
- API contracts can break consumers
- Determinism checks prevent flaky deployments

In [ ]:
# Example deterministic check idea
sample_input = {'feature_1': 1.0, 'feature_2': 2.0}
# Replace with your model call once implemented
pred_a = 0.75
pred_b = 0.75
assert pred_a == pred_b, 'Prediction should be deterministic for same input in this demo'
print('Determinism check passed')

## 6) Containerize the Service with Docker

Concepts:
- Base image choice affects security and size
- Layer caching speeds up rebuilds
- Pinned dependencies reduce drift
- Images are immutable deployment artifacts

Typical flow:
1. Build image
2. Tag image
3. Run image locally
4. Push image to registry

In [ ]:
# Lightweight image tag helper
from datetime import datetime
tag = datetime.utcnow().strftime('v%Y%m%d-%H%M%S')
print('Example image tag:', tag)
print('Build cmd: docker build -t my-infer-api:' + tag + ' .')

## 7) Run Multi-Service Local Development with Docker Compose

Compose gives local parity:
- App service
- Optional test runner service
- Shared network and env variables

Knowledge check:
1. Why does local parity reduce deployment surprises?
2. What should differ between local and production configs?

## 8) Implement CI Pipeline for Test, Lint, and Image Build

CI gatekeeping principle:
- Merge only if checks pass
- Fail fast on lint/tests
- Build image as a final confidence step

Benefits:
- Lower regression risk
- Better team velocity
- Reproducible quality checks

In [ ]:
# Tiny YAML-like structure demo for pipeline steps
ci_steps = ['checkout', 'setup-python', 'install-deps', 'lint', 'test', 'docker-build']
for i, step in enumerate(ci_steps, start=1):
    print(f'{i}. {step}')

## 9) Implement CD Pipeline for Cloud Deployment

CD responsibilities:
- Promote verified artifacts
- Push immutable tags to registry
- Deploy by environment (dev -> staging -> prod)

Cloud-neutral path:
- Build once
- Push to registry once
- Deploy same image digest across environments

Knowledge check:
1. Why use immutable tags and digests?
2. What is artifact promotion?

## 10) Manage Configuration, Secrets, and Environments

Golden rule: separate code from configuration.

Use:
- `.env` style values for non-secret config
- Secret managers for credentials
- Least-privilege service accounts

Never commit raw cloud keys to git.

Knowledge check:
1. Which values belong in config vs secret manager?
2. Why should each environment have isolated credentials?

In [ ]:
import os

# Demo only: in production, read secrets from cloud secret manager.
os.environ['APP_ENV'] = os.getenv('APP_ENV', 'dev')
print('Current environment:', os.environ['APP_ENV'])

## 11) Add Observability, Health Checks, and Rollback Workflow

You need release safety signals:
- `/health` endpoint for readiness/liveness
- Structured logs for debugging and audits
- Error-rate and latency monitoring

Use error rate as a release gate:
$$
\text{Error Rate} = \frac{\text{Failed Requests}}{\text{Total Requests}}
$$

If error rate spikes after rollout:
1. Halt rollout
2. Roll back to previous image digest
3. Open incident notes and root cause investigation

In [ ]:
# Failure simulation for rollback decision
failed_requests = 23
total_requests = 400
error_rate = failed_requests / total_requests
print('Error rate:', round(error_rate, 4))
if error_rate > 0.03:
    print('Action: rollback recommended')
else:
    print('Action: continue rollout')

## Lab Mapping to Starter Files

Implement your code inside marker blocks only.

- API app: `app/main.py`
- Model loader: `app/model_loader.py`
- Tests: `tests/test_api.py`
- Dependencies: `requirements.txt`
- Container: `Dockerfile`
- Local orchestration: `docker-compose.yml`
- CI: `.github/workflows/ci.yml`
- CD: `.github/workflows/cd.yml`
- Cloud deployment template: `cloud/deploy/cloudrun-service.yaml`
- Environment configs: `cloud/config/dev.env`, `cloud/config/staging.env`, `cloud/config/prod.env`

Success criteria:
- `pytest` passes
- `docker build` succeeds
- `docker compose up` starts service
- CI workflow is green
- CD workflow deploys to your selected cloud target

## Capstone Project

Goal:
- Deploy one model-serving API from this repo to a cloud-managed container service.

Capstone deliverables:
- Versioned artifact + manifest
- Tested API
- Docker image in registry
- Green CI and CD workflows
- Health checks + basic rollback note

Reflection prompts:
1. Which part was hardest: API, tests, Docker, CI, or CD?
2. Which production risk remains and how would you reduce it?

## 30-Day Study Plan

Week 1:
- Complete API + model loader + tests
- Learn schema validation and deterministic checks

Week 2:
- Complete Docker and Compose setup
- Measure startup time and request latency locally

Week 3:
- Implement CI workflow and enforce quality gates
- Add code style and test coverage checks

Week 4:
- Implement CD workflow to one cloud target
- Add health checks, monitoring baseline, and rollback drill

After 30 days:
- Repeat with a second model
- Add canary rollout strategy
- Add load testing and cost analysis